# 19 · Multi-Query：一个问题，多路召回

> 一个“说法”可能错过很多相关内容。让 LLM 从不同角度改写出多个查询，各自召回再融合，召回率明显提升。

**本文件覆盖知识点**：Multi-Query / Parallel Retrieval / 多查询生成

```text
"Redis为什么快?"
  → "Redis性能为什么高？"
  → "Redis为什么采用单线程？"
  → "Redis IO模型是什么？"
  → "Redis为什么比MySQL快？"      ← 各自检索，合并去重
```

In [ ]:
# ===== 本课共用：真实检索底座 =====
# 真语料(data/) → 真切分 → 真向量(text-embedding-v3) → 真索引(FAISS + BM25)
# → 真重排(qwen3-rerank) → 真生成(qwen-plus)。各课在这个底座上演示自己的知识点。
#
# 说明：向量按内容哈希缓存在 .cache/emb.npz（首次真调、之后复用，避免反复花 token）。
# 没配 DASHSCOPE_API_KEY 时仍可用：向量直接从缓存读（是此前真实调用的结果），
# 但需要现场调用模型的重排/生成会打印录制结果并提示配置方式。
from dotenv import load_dotenv; load_dotenv()
import os, re, json, time, hashlib
from pathlib import Path
import numpy as np

_KEY = os.getenv('DASHSCOPE_API_KEY', '').strip()
_HAS_KEY = bool(_KEY) and '你的' not in _KEY
_DATA = Path('data') if Path('data').is_dir() else Path.cwd() / 'data'
_CACHE_FILE = Path('.cache') / 'emb.npz'
EMBED_MODEL = 'text-embedding-v3'
RERANK_MODEL = 'qwen3-rerank'
NO_KEY_TIP = ('未配置 DASHSCOPE_API_KEY：需要现场调用模型的部分将展示此前真实调用的录制结果，'
              '在项目根 .env 配置后自动变为实时调用。')

def recorded(text, note=''):
    """无 Key 时展示「此前真实运行的录制结果」。内容来自真实调用，不是编造的假数据。"""
    print(NO_KEY_TIP)
    print('—— 录制结果%s ——' % ('（' + note + '）' if note else ''))
    print(text)

if not _HAS_KEY:
    print(NO_KEY_TIP)

# ---------- 1) 语料：读 data/ 全部 Markdown，按小节切块 ----------
# 注意：评测集*.md 是「人工标注的答案」，不能进索引 —— 否则第 34 课评测时，
# 标注本身会被检索命中，指标虚高（数据泄漏）。这里按文件名前缀排除（含第 33 课产出的 评测集_v2.md）。
_EXCLUDE_PREFIX = '评测集'

def load_chunks(chunk_size=300, overlap=60):
    """按「## 小节」切分，小节过长再按句子窗口滑切。返回 [{'i','text','source','section'}]"""
    out = []
    for p in sorted(_DATA.glob('*.md')):
        if p.name.startswith(_EXCLUDE_PREFIX):
            continue
        section, buf = p.stem, []
        for line in p.read_text(encoding='utf-8').splitlines():
            if line.startswith('## '):
                if buf: out += _split_section(buf, section, p.name, chunk_size, overlap)
                section, buf = line[3:].strip(), [line]
            elif line.startswith('# '):
                section = line[2:].strip()
            else:
                buf.append(line)
        if buf: out += _split_section(buf, section, p.name, chunk_size, overlap)
    for i, c in enumerate(out):
        c['i'] = i
    return out

def _split_section(lines, section, source, chunk_size, overlap):
    """小节内容按句号聚合成 ~chunk_size 字的片段，相邻片段留 overlap 字重叠"""
    text = '\n'.join(lines).strip()
    if not text: return []
    sents = [s for s in re.split(r'(?<=[。！？\n])', text) if s.strip()]
    chunks, buf = [], ''
    for s in sents:
        if len(buf) + len(s) > chunk_size and buf:
            chunks.append(buf.strip())
            buf = buf[-overlap:] + s          # 保留尾部 overlap 字做上下文重叠
        else:
            buf += s
    if buf.strip(): chunks.append(buf.strip())
    return [{'text': c, 'source': source, 'section': section} for c in chunks]

# ---------- 2) 向量：真调 text-embedding-v3（分批 + 重试 + 内容哈希缓存）----------
def _load_cache():
    if not _CACHE_FILE.exists():
        return {}
    try:
        z = np.load(_CACHE_FILE, allow_pickle=False)
        return dict(zip(z['hashes'].tolist(), z['vectors']))
    except Exception as e:                      # 文件损坏（例如多进程同时写）：当空缓存重建，别让 notebook 挂掉
        print('向量缓存不可读(%s: %s)，将重新向量化：%s' % (type(e).__name__, e, _CACHE_FILE))
        return {}

def _save_cache(cache):
    """写盘前先与磁盘上已有内容合并，再原子替换 —— 避免多个进程同时跑时互相覆盖 / 写坏文件"""
    _CACHE_FILE.parent.mkdir(parents=True, exist_ok=True)
    for k, v in _load_cache().items():
        cache.setdefault(k, v)
    hs = np.array(list(cache.keys()))
    vs = np.array([cache[h] for h in cache.keys()], dtype='float32')
    # 进程号唯一，别抢同一个临时文件；注意 np.savez_compressed 会自动补 .npz 后缀，临时名必须也是 .npz 结尾
    tmp = _CACHE_FILE.with_name('%s.%d.tmp.npz' % (_CACHE_FILE.stem, os.getpid()))
    np.savez_compressed(tmp, hashes=hs, vectors=vs)
    try:
        os.replace(tmp, _CACHE_FILE)            # 原子替换：别的进程读到的永远是完整文件
    except OSError:                             # 目标被占用时稍等再试
        time.sleep(0.2); os.replace(tmp, _CACHE_FILE)

def _key(text, model):
    return hashlib.sha1((model + '\x00' + text).encode('utf-8')).hexdigest()[:16]

def embed(texts, model=EMBED_MODEL, batch=10):
    """真调 Embedding；命中缓存则直接用（缓存来自真实调用）。返回已 L2 归一化的向量"""
    if isinstance(texts, str): texts = [texts]
    cache, todo = _load_cache(), []
    for t in texts:
        k = _key(t, model)
        if k not in cache and k not in [x[0] for x in todo]:
            todo.append((k, t))
    if todo and not _HAS_KEY:
        raise RuntimeError('本地缓存缺少 %d 条向量，且未配置 DASHSCOPE_API_KEY，无法现场向量化。'
                           '请在项目根 .env 配置 Key 后重跑。' % len(todo))
    if todo:
        from dashscope import TextEmbedding
        pending = todo
        while pending:                                  # 批次过大就减半重试
            b = pending[:batch]
            r = TextEmbedding.call(model=model, input=[t for _, t in b], api_key=_KEY)
            if r.status_code == 200:
                for (k, _), e in zip(b, sorted(r.output['embeddings'], key=lambda e: e['text_index'])):
                    cache[k] = np.array(e['embedding'], dtype='float32')
                pending = pending[len(b):]
            elif batch > 1:
                batch //= 2
            else:
                raise RuntimeError('向量化失败: %s %s' % (r.code, r.message))
        _save_cache(cache)
    v = np.array([cache[_key(t, model)] for t in texts], dtype='float32')
    return v / (np.linalg.norm(v, axis=1, keepdims=True) + 1e-10)

# ---------- 3) 索引：FAISS（归一化后内积=余弦）+ BM25 ----------
import faiss
from rank_bm25 import BM25Okapi

def tokenize(text):
    """中文用「单字 + 相邻双字」切词，无需外部分词器（与第 16 课一致）"""
    t = re.sub(r'\s+', '', text)
    return [t[i] for i in range(len(t))] + [t[i:i + 2] for i in range(len(t) - 1)]

CHUNKS = load_chunks()
VECS = embed([c['text'] for c in CHUNKS])
INDEX = faiss.IndexFlatIP(VECS.shape[1]); INDEX.add(VECS)
BM25 = BM25Okapi([tokenize(c['text']) for c in CHUNKS])
print('语料就绪：%d 篇文档 → %d 个片段，向量维度 %d' % (len({c['source'] for c in CHUNKS}), len(CHUNKS), VECS.shape[1]))

# ---------- 4) 检索：稠密 / 稀疏 / 混合（RRF 融合）----------
def dense_retrieve(query, k=5):
    sims, ids = INDEX.search(embed(query), k)
    return [dict(CHUNKS[i], score=float(s), from_='dense') for i, s in zip(ids[0], sims[0]) if i != -1]

def sparse_retrieve(query, k=5):
    scores = BM25.get_scores(tokenize(query))
    top = np.argsort(-scores)[:k]
    return [dict(CHUNKS[i], score=float(scores[i]), from_='bm25') for i in top if scores[i] > 0]

def hybrid_retrieve(query, k=5, rrf_k=60, pool=10):
    """RRF 融合：score = Σ 1/(rrf_k + rank)，只用名次不用原始分数，天然可比"""
    fused = {}
    for name, hits in (('dense', dense_retrieve(query, pool)), ('bm25', sparse_retrieve(query, pool))):
        for rank, h in enumerate(hits, 1):
            cur = fused.setdefault(h['i'], dict(h, score=0.0, from_=set()))
            cur['score'] += 1.0 / (rrf_k + rank)
            cur['from_'].add(name)
    return sorted(fused.values(), key=lambda x: -x['score'])[:k]

# ---------- 5) 重排：真调 DashScope TextReRank ----------
def rerank(query, docs, top_n=3, model=RERANK_MODEL):
    """docs 可以是字符串列表或检索结果 dict 列表；返回 [(文档, 相关性分数)]"""
    texts = [d['text'] if isinstance(d, dict) else d for d in docs]
    if not texts: return []
    if not _HAS_KEY:
        print(NO_KEY_TIP); return [(t, None) for t in texts[:top_n]]
    from dashscope import TextReRank
    r = TextReRank.call(model=model, query=query, documents=texts,
                        top_n=min(top_n, len(texts)), return_documents=False, api_key=_KEY)
    if r.status_code != 200:
        raise RuntimeError('重排失败: %s %s' % (r.code, r.message))
    return [(texts[it['index']], float(it['relevance_score'])) for it in r.output['results']]

# ---------- 6) 生成：qwen-plus（带重试）+ 结构化 JSON 输出 ----------
def chat(prompt, system='你是严谨的 RAG 助手：只依据给定资料回答，资料里没有的就直说不知道。',
         temperature=0.3, model='qwen-plus', retries=3):
    if not _HAS_KEY:
        return None
    from dashscope import Generation
    for attempt in range(retries):
        r = Generation.call(model=model, messages=[{'role': 'system', 'content': system},
                                                   {'role': 'user', 'content': prompt}],
                            temperature=temperature, result_format='message', api_key=_KEY)
        if r.status_code == 200:
            return r.output.choices[0].message.content
        if attempt == retries - 1:
            raise RuntimeError('生成失败: %s %s' % (r.code, r.message))
        time.sleep(1.5 * (attempt + 1))          # 限流类错误退避重试
    return None

def chat_json(prompt, system='只输出 JSON，不要任何解释或代码块标记。', retries=2, **kw):
    """要求模型输出 JSON 并解析；解析失败时把报错回喂再试一次"""
    for attempt in range(retries + 1):
        out = chat(prompt, system=system, **kw)
        if out is None: return None
        seg = out[out.find('{'): out.rfind('}') + 1]     # 容忍 ```json 包裹与前后废话
        try:
            return json.loads(seg)
        except Exception as e:
            if attempt == retries: raise
            prompt = prompt + '\n\n上次输出无法解析(%s)，请只输出合法 JSON。' % e
    return None


In [ ]:
# .env 配置（DASHSCOPE_API_KEY 配在项目根 .env，由下面的底座 cell 统一读取）
# 多查询生成：让 qwen-plus 就同一个原始问题改写出 N 个「侧重不同」的检索查询
QUERY, N = '星云客服机器人怎么收费', 3

def generate_queries(query, n=3):
    """把 query 改写成 n 个意思相关、侧重不同的检索查询（原查询保留，一起进检索）。
    结构化输出交给底座的 chat_json：它负责剥掉代码块/废话并解析 JSON，失败还会回喂重试。"""
    obj = chat_json(
        '把下面问题改写成 %d 个意思相关但侧重不同的检索查询。\n'
        '只输出 JSON 对象：{"queries": ["改写1", "..."]}\n问题: %s' % (n, query),
        system='你是检索查询改写器：只输出 JSON，不要解释。')
    got = obj.get('queries', []) if isinstance(obj, dict) else []
    return [query] + [str(q) for q in got[:n]]

if _HAS_KEY:
    QUERIES = generate_queries(QUERY, N)
    print('原始问题：%s' % QUERY)
    print('qwen-plus 改写出的 %d 个查询：' % (len(QUERIES) - 1))
    for i, q in enumerate(QUERIES[1:], 1):
        print('  %d. %s' % (i, q))
else:
    recorded("""原始问题：星云客服机器人怎么收费
qwen-plus 改写出的 3 个查询：
  1. 星云客服机器人收费标准及价格方案
  2. 星云客服机器人按什么计费（如按坐席/按消息量/包年）
  3. 星云客服机器人免费版与付费版功能对比及费用详情""",
             '录制于 2026-09-12，模型 qwen-plus')
    # 录制内容里的真实改写结果，供下面的检索/融合继续用（无 Key 时检索本身仍真跑，向量走缓存）
    QUERIES = [QUERY, '星云客服机器人收费标准及价格方案',
               '星云客服机器人按什么计费（如按坐席/按消息量/包年）',
               '星云客服机器人免费版与付费版功能对比及费用详情']


## 检索与融合

Multi-Query 的检索与融合和普通检索一致：
```text
q1 ──→ 检索 ──┐
q2 ──→ 检索 ──┼─→ 合并(去重/按名次 RRF) ──→ 候选池
q3 ──→ 检索 ──┘
```

- **合并手段**：简单拼接去重，或按 17 课的 RRF 融合多个榜单；
- **并行执行**：多个查询相互独立，应并发请求以省时（`concurrent.futures` / 异步）；
- **代价**：检索次数变成 n 倍，延迟与成本上升，注意控制 n 与候选池大小。

In [ ]:
# 多路召回 → RRF 融合（rrf_fuse 与 17 课同一个函数，公式不动；输入换成每路真实检索结果）
def rrf_fuse(rankings, k=60):
    score = {}
    for ranking in rankings:
        for rank, doc in enumerate(ranking):
            score[doc] = score.get(doc, 0) + 1.0 / (k + rank + 1)
    return sorted(score.items(), key=lambda x: -x[1])

def _lab(h):
    return '%s·%s' % (h['source'].replace('.md', ''), h['section'])

K, TOPK = 5, 5                                  # 每路取 K 个候选进池，融合后输出 Top-k
rankings, labels = [], {}
for qi, q in enumerate(QUERIES):
    hits = hybrid_retrieve(q, k=K)              # 每个查询都真跑一次混合检索（内部已做 dense+BM25 的 RRF）
    rankings.append([h['i'] for h in hits])
    for h in hits:
        labels[h['i']] = _lab(h)
    print('路%d｜%s' % (qi, q))
    for r, h in enumerate(hits, 1):
        print('    %d. %-32s %.6f  %s' % (r, _lab(h), h['score'], h['text'][:20].replace('\n', ' ')))

fused = rrf_fuse(rankings)
pos = {doc: r for r, (doc, _) in enumerate(fused, 1)}
print('\nRRF 融合后的最终 Top-%d（多路名次分相加，被多路共同命中的更靠前）:' % TOPK)
for r, (doc, s) in enumerate(fused[:TOPK], 1):
    routes = [i for i, L in enumerate(rankings) if doc in L]
    print('   %d. %-32s %.6f  被 %d 路召回(路%s)' % (r, labels[doc], s, len(routes), routes))

# 对照：只看原查询 vs 多路融合 —— 「召回提升」要落到本次真实数据上，不能只喊口号
base_ids = set(rankings[0])
union_ids = set().union(*[set(L) for L in rankings])
pool_new = sorted(union_ids - base_ids, key=lambda d: pos[d])
print('\n对照（原查询单独 vs %d 路）:' % len(rankings))
print('  只用原查询：候选 %d 条，Top-%d = %s'
      % (len(base_ids), TOPK, [labels[d] for d in rankings[0][:TOPK]]))
print('  %d 路候选池去重后 %d 条，改写新捞到 %d 条：%s'
      % (len(rankings), len(union_ids), len(pool_new),
         ', '.join('%s(路%s)' % (labels[d], [i for i, L in enumerate(rankings) if d in L]) for d in pool_new) or '（无）'))
print('  名次变化（融合 vs 原查询）:')
for r, (doc, _) in enumerate(fused[:TOPK], 1):
    old = rankings[0].index(doc) + 1 if doc in rankings[0] else None
    print('    %d. %-32s %s' % (r, labels[doc],
          '原查询第 %d 位 → 融合第 %d 位' % (old, r) if old else '原查询没进 Top-%d → 融合第 %d 位' % (K, r)))
gain = [labels[d] for d, _ in fused[:TOPK] if d not in base_ids]
print('→ ' + ('改写把新片段顶进了最终 Top-%d：%s' % (TOPK, ', '.join(gain)) if gain else
      '本次改写没有新片段挤进最终 Top-%d：单路独有的命中在 RRF 里排在多路共同命中之后，'
      '但它确实把候选池从 %d 条扩到 %d 条（新增 %s），这些片段会进入后续 Rerank 的候选；'
      '同时多路投票让被多路召回的片段名次上升（见上面的名次变化）。'
      % (TOPK, len(base_ids), len(union_ids), ', '.join(labels[d] for d in pool_new) or '无')))
print('（生产上各路检索相互独立，可用线程池并发以省时；融合后仍要接 Rerank 做最终精排。）')


## 小结

- Multi-Query = 多角度改写 + 并行召回 + 融合去重；
- 融合建议 RRF，召回建议并发；
- 与 RAG-Fusion（下一课）是同一思路加结构化组装。

如果连“扩展出的查询”本身也想优化（同义词加权、按查询加权融合），就到了 RAG-Fusion 与 Query Expansion。